Imports

In [ ]:
# Imports
import json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

Data Loading

In [ ]:
# Define paths
TREATMENT_DIR = Path("timelines/pa_treatment_timelines_clean")
CONTROL_DIR = Path("timelines/pa_control_timelines_clean")

In [ ]:
# Load timelines function
def load_timelines(directory, group_label):
    # a function to load all timelines from a directory and assign a group label
    timeline_files = sorted(directory.glob("*.json"))
    print(f"Found {len(timeline_files)} timelines in {directory}")
    
    timelines_data = []
    for f in timeline_files:
        with open(f, 'r') as infile:
            data = json.load(infile)
            data['group'] = group_label
            timelines_data.append(data)
    return timelines_data

In [ ]:
# Load both datasets
treatment_timelines = load_timelines(TREATMENT_DIR, 'treatment')
control_timelines = load_timelines(CONTROL_DIR, 'control')

Shared Functions

In [ ]:
def analyze_timeline_group(timelines, group_name=""):
    # a function to perform and print basic EDA on a list of timelines
    
    # 1. Basic Stats
    stats = []
    for timeline in timelines:
        stats.append({
            'patient_id': timeline['patient_id'],
            'num_events': len(timeline['events']),
            'date_start': timeline['date_range']['start'],
            'date_end': timeline['date_range']['end']
        })
    df_stats = pd.DataFrame(stats)
    
    if df_stats.empty:
        print(f"--- Analysis for {group_name} Group ---")
        print("No data to analyze.")
        return None, None, None
        
    # Coerce errors will turn bad dates into NaT (Not a Time)
    df_stats['date_start_dt'] = pd.to_datetime(df_stats['date_start'], errors='coerce')
    df_stats['date_end_dt'] = pd.to_datetime(df_stats['date_end'], errors='coerce')
    df_stats['duration_days'] = (df_stats['date_end_dt'] - df_stats['date_start_dt']).dt.days

    print(f"--- Analysis for {group_name} Group ---")
    print(f"\nTotal patients: {len(df_stats)}")
    
    print(f"\nEvent Count Statistics:")
    print(df_stats['num_events'].describe())
    print(f"\nDuration Statistics (days):")
    print(df_stats['duration_days'].describe())

    # 2. Concept Stats
    cui_counter = Counter()
    cui_to_name = {}
    for timeline in timelines:
        for event in timeline['events']:
            cui = event['entity_cui']
            cui_counter[cui] += 1
            cui_to_name[cui] = event['entity_preferred_name']
            
    print(f"\nTotal events: {sum(cui_counter.values())}")
    print(f"Unique concepts (CUIs): {len(cui_counter)}")
    
    print(f"\nTop 10 most frequent concepts:")
    for cui, count in cui_counter.most_common(10):
        print(f"  CUI {cui}: {count} occurrences ({cui_to_name.get(cui, 'Unknown')})")

    return df_stats, cui_counter, cui_to_name

In [ ]:
def plot_timeline_length_distribution(df_stats, group_name=""):
    # a function to plot the distribution of timeline lengths
    if df_stats is None or df_stats.empty:
        print(f"No data to plot for {group_name} group.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f"Timeline Length Distribution for {group_name} Group", fontsize=16)

    # Histogram
    axes[0].hist(df_stats['num_events'], bins=30, edgecolor='black')
    axes[0].set_xlabel('Number of Events')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Timeline Lengths')
    axes[0].axvline(df_stats['num_events'].mean(), color='r', linestyle='--', 
                    label=f'Mean: {df_stats["num_events"].mean():.1f}')
    axes[0].axvline(df_stats['num_events'].median(), color='g', linestyle='--', 
                    label=f'Median: {df_stats["num_events"].median():.1f}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Boxplot
    axes[1].boxplot(df_stats['num_events'], vert=True)
    axes[1].set_ylabel('Number of Events')
    axes[1].set_title('Timeline Length Boxplot')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
def plot_top_concepts(cui_counter, cui_to_name, group_name=""):
    # a function to plot a bar chart of the top 15 concepts
    if cui_counter is None or not cui_counter:
        print(f"No CUI data to plot for {group_name} group.")
        return

    top_concepts = cui_counter.most_common(15)
    cui_ids = [cui for cui, _ in top_concepts]
    counts = [count for _, count in top_concepts]
    labels = [cui_to_name.get(cui, f"CUI {cui}") for cui in cui_ids]

    # Reverse order so highest is at top
    labels = labels[::-1]
    counts = counts[::-1]

    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.barh(range(len(top_concepts)), counts, edgecolor='black')
    ax.set_yticks(range(len(top_concepts)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Frequency (Number of Occurrences)')
    ax.set_title(f'Top 15 Most Frequent Concepts (CUIs) in {group_name} Group')
    ax.grid(True, alpha=0.3, axis='x')

    # Add value labels on bars
    for i, v in enumerate(counts):
        ax.text(v + 1, i, str(v), va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
def analyze_and_plot_diversity(timelines, group_name=""):
    # a function to analyze and plot concept diversity
    if not timelines:
        print(f"No timelines to analyze for {group_name} group.")
        return

    patient_concept_stats = []
    for timeline in timelines:
        patient_cuis = [event['entity_cui'] for event in timeline['events']]
        if not patient_cuis:
            continue
        
        unique_cuis = set(patient_cuis)
        patient_concept_stats.append({
            'patient_id': timeline['patient_id'],
            'total_concepts': len(patient_cuis),
            'unique_concepts': len(unique_cuis),
        })

    df_concept_stats = pd.DataFrame(patient_concept_stats)
    if df_concept_stats.empty:
        print(f"No concept data to plot for {group_name} group.")
        return
    
    df_concept_stats['diversity_ratio'] = df_concept_stats['unique_concepts'] / df_concept_stats['total_concepts']

    # Create Plots
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Concept Diversity Analysis for {group_name} Group", fontsize=16)

    # Scatter plot
    sns.scatterplot(data=df_concept_stats, x='total_concepts', y='unique_concepts', alpha=0.6, ax=axes[0])
    axes[0].set_title('Concept Diversity per Patient')
    axes[0].set_xlabel('Total Concepts (Timeline Length)')
    axes[0].set_ylabel('Unique Concepts')
    max_val = int(df_concept_stats['total_concepts'].max()) if not df_concept_stats.empty else 1
    axes[0].plot([0, max_val], [0, max_val], color='red', linestyle='--', label='Perfect Diversity (y=x)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Histogram
    sns.histplot(data=df_concept_stats, x='diversity_ratio', bins=20, kde=True, ax=axes[1])
    axes[1].set_title('Distribution of Patient Diversity Ratios')
    axes[1].set_xlabel('Diversity Ratio (Unique / Total Concepts)')
    axes[1].set_ylabel('Number of Patients')
    axes[1].axvline(df_concept_stats['diversity_ratio'].mean(), color='r', linestyle='--', label=f"Mean: {df_concept_stats['diversity_ratio'].mean():.2f}")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # Display summary statistics
    print(df_concept_stats[['total_concepts', 'unique_concepts', 'diversity_ratio']].describe())

Exploration of Treatment Group

In [ ]:
# Analyze the treatment group
df_stats_treat, cui_counter_treat, cui_to_name_treat = analyze_timeline_group(treatment_timelines, "Treatment")

In [ ]:
# Plot the timeline length distribution for the treatment group
plot_timeline_length_distribution(df_stats_treat, "Treatment")

In [ ]:
# Plot the top concepts for the treatment group
plot_top_concepts(cui_counter_treat, cui_to_name_treat, "Treatment")

In [ ]:
# Analyze and plot concept diversity for the treatment group
analyze_and_plot_diversity(treatment_timelines, "Treatment")

Exploration of Control Group

In [ ]:
# Analyze the control group
df_stats_control, cui_counter_control, cui_to_name_control = analyze_timeline_group(control_timelines, "Control")

In [ ]:
# Plot the timeline length distribution for the control group
plot_timeline_length_distribution(df_stats_control, "Control")

In [ ]:
# Plot the top concepts for the control group
plot_top_concepts(cui_counter_control, cui_to_name_control, "Control")

In [ ]:
# Analyze and plot concept diversity for the control group
analyze_and_plot_diversity(control_timelines, "Control")

Comparison of Groups

In [ ]:
#Compare timeline length and duration

# Combine the statistics DataFrames for easier plotting
df_stats_treat['group'] = 'Treatment'
df_stats_control['group'] = 'Control'
df_stats_combined = pd.concat([df_stats_treat, df_stats_control], ignore_index=True)

# Plot the distributions for comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle("Comparison of Timeline Metrics by Group", fontsize=16)

# Comparison of Number of Events
sns.boxplot(data=df_stats_combined, x='group', y='num_events', ax=axes[0])
axes[0].set_title('Distribution of Timeline Lengths (Number of Events)')
axes[0].set_xlabel('Group')
axes[0].set_ylabel('Number of Events')
axes[0].grid(True, alpha=0.3)

# Comparison of Duration in Days
sns.boxplot(data=df_stats_combined, x='group', y='duration_days', ax=axes[1])
axes[1].set_title('Distribution of Timeline Durations')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('Duration (Days)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Print summary statistics for comparison
print(df_stats_combined.groupby('group')[['num_events', 'duration_days']].describe())

In [ ]:
# Analyze the overlap of concepts between the two groups
treatment_concepts = set(cui_counter_treat.keys())
control_concepts = set(cui_counter_control.keys())

# Calculate intersections and differences
shared_concepts = treatment_concepts.intersection(control_concepts)
unique_to_treatment = treatment_concepts - control_concepts
unique_to_control = control_concepts - treatment_concepts

print("--- Concept Overlap Analysis ---")
print(f"Total unique concepts in Treatment group: {len(treatment_concepts)}")
print(f"Total unique concepts in Control group:   {len(control_concepts)}")
print(f"Number of concepts shared between groups: {len(shared_concepts)}")
print(f"Number of concepts unique to Treatment:   {len(unique_to_treatment)}")
print(f"Number of concepts unique to Control:     {len(unique_to_control)}")

In [ ]:
# Create a combined DataFrame of CUI counts for comparison
df_treat_counts = pd.DataFrame(cui_counter_treat.most_common(), columns=['cui', 'count_treat'])
df_control_counts = pd.DataFrame(cui_counter_control.most_common(), columns=['cui', 'count_control'])

# Merge the dataframes to align counts by CUI
df_counts_merged = pd.merge(df_treat_counts, df_control_counts, on='cui', how='outer').fillna(0)

# Calculate total occurrences and frequency per group
total_treat_events = sum(cui_counter_treat.values())
total_control_events = sum(cui_counter_control.values())
df_counts_merged['freq_treat'] = df_counts_merged['count_treat'] / total_treat_events
df_counts_merged['freq_control'] = df_counts_merged['count_control'] / total_control_events

# Calculate a distinctiveness score (log ratio of frequencies)
# We add a small epsilon to avoid division by zero
epsilon = 1e-9
df_counts_merged['distinctiveness'] = np.log2((df_counts_merged['freq_treat'] + epsilon) / (df_counts_merged['freq_control'] + epsilon))

# Get the top 15 most distinctive concepts for each group
df_distinct_treat = df_counts_merged.sort_values(by='distinctiveness', ascending=False).head(15)
df_distinct_control = df_counts_merged.sort_values(by='distinctiveness', ascending=True).head(15)

# Combine all CUI names for the plot labels
all_cui_names = {**cui_to_name_treat, **cui_to_name_control}

def plot_distinctive_concepts(df_distinct, group_name, all_cui_names):
    # a function to plot the most distinctive concepts
    df_distinct['label'] = df_distinct['cui'].apply(lambda x: all_cui_names.get(x, f'CUI {x}'))
    
    fig, ax = plt.subplots(figsize=(12, 7))
    colors = ['#3498db' if x > 0 else '#e74c3c' for x in df_distinct['distinctiveness']]
    
    bars = ax.barh(df_distinct['label'], df_distinct['distinctiveness'], color=colors)
    ax.set_xlabel('Distinctiveness (Log2 Frequency Ratio)')
    ax.set_title(f'Top 15 Concepts Most Distinctive to the {group_name} Group')
    ax.grid(True, alpha=0.3, axis='x')
    ax.axvline(0, color='black', linewidth=0.8) # Add a line at zero

    plt.tight_layout()
    plt.show()

# Plot for Treatment Group
plot_distinctive_concepts(df_distinct_treat.iloc[::-1], 'Treatment', all_cui_names)

# Plot for Control Group
plot_distinctive_concepts(df_distinct_control, 'Control', all_cui_names)

In [ ]:
def get_diversity_df(timelines, group_name):
    # a helper function to calculate diversity stats
    patient_concept_stats = []
    for timeline in timelines:
        patient_cuis = [event['entity_cui'] for event in timeline['events']]
        if not patient_cuis:
            continue
        unique_cuis = set(patient_cuis)
        patient_concept_stats.append({
            'patient_id': timeline['patient_id'],
            'total_concepts': len(patient_cuis),
            'unique_concepts': len(unique_cuis),
        })
    df = pd.DataFrame(patient_concept_stats)
    df['diversity_ratio'] = df['unique_concepts'] / df['total_concepts']
    df['group'] = group_name
    return df

# Get diversity data for both groups
df_diversity_treat = get_diversity_df(treatment_timelines, 'Treatment')
df_diversity_control = get_diversity_df(control_timelines, 'Control')
df_diversity_combined = pd.concat([df_diversity_treat, df_diversity_control], ignore_index=True)

# Plot the distribution of diversity ratios
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df_diversity_combined, x='diversity_ratio', hue='group', fill=True, common_norm=False)
plt.title('Comparison of Concept Diversity Ratios by Group')
plt.xlabel('Diversity Ratio (Unique / Total Concepts)')
plt.ylabel('Density')
plt.grid(True, alpha=0.3)
plt.axvline(df_diversity_treat['diversity_ratio'].mean(), color='blue', linestyle='--', label=f"Mean Treatment: {df_diversity_treat['diversity_ratio'].mean():.2f}")
plt.axvline(df_diversity_control['diversity_ratio'].mean(), color='orange', linestyle='--', label=f"Mean Control: {df_diversity_control['diversity_ratio'].mean():.2f}")
plt.legend()
plt.show()

# Print summary statistics
print(df_diversity_combined.groupby('group')['diversity_ratio'].describe())

In [ ]:
# Create a side-by-side table of top concepts, similar to a feature importance table

# Get top 20 concepts for each group
top_treat = cui_counter_treat.most_common(20)
top_control = cui_counter_control.most_common(20)

# Create dataframes
df_top_treat = pd.DataFrame(top_treat, columns=['Treatment CUI', 'Treatment Count'])
df_top_control = pd.DataFrame(top_control, columns=['Control CUI', 'Control Count'])

# Add concept names
df_top_treat['Treatment Concept'] = df_top_treat['Treatment CUI'].apply(lambda x: cui_to_name_treat.get(x, 'Unknown'))
df_top_control['Control Concept'] = df_top_control['Control CUI'].apply(lambda x: cui_to_name_control.get(x, 'Unknown'))

# Reorder columns for readability
df_top_treat = df_top_treat[['Treatment Concept', 'Treatment Count', 'Treatment CUI']]
df_top_control = df_top_control[['Control Concept', 'Control Count', 'Control CUI']]

# Concatenate for side-by-side view
df_comparison = pd.concat([df_top_treat.reset_index(drop=True), df_top_control.reset_index(drop=True)], axis=1)

print("--- Top 20 Most Frequent Concepts: Side-by-Side Comparison ---")
display(df_comparison)

In [ ]:
# Calculate Odds Ratios for each concept to identify associations with the treatment group

# 1. Get total patient counts for each group
n_treatment = len(treatment_timelines)
n_control = len(control_timelines)

# 2. Get patient-level presence for each CUI (i.e., how many patients have the concept)
def get_patient_presence_counts(timelines):
    # a function to count how many patients have at least one instance of a CUI
    cui_patient_counter = Counter()
    for timeline in timelines:
        # Get unique CUIs for this patient to avoid double-counting
        unique_cuis_in_timeline = {event['entity_cui'] for event in timeline['events']}
        for cui in unique_cuis_in_timeline:
            cui_patient_counter[cui] += 1
    return cui_patient_counter

treat_patient_counts = get_patient_presence_counts(treatment_timelines)
control_patient_counts = get_patient_presence_counts(control_timelines)

# 3. Calculate odds ratio for each concept in the combined vocabulary
all_cuis = set(treat_patient_counts.keys()) | set(control_patient_counts.keys())
odds_ratios = []

for cui in all_cuis:
    # Contingency table values
    a = treat_patient_counts.get(cui, 0)  # Treatment patients with CUI
    b = n_treatment - a                   # Treatment patients without CUI
    c = control_patient_counts.get(cui, 0) # Control patients with CUI
    d = n_control - c                     # Control patients without CUI
    
    # Apply continuity correction to avoid division by zero
    a_corr, b_corr, c_corr, d_corr = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    
    # Calculate odds ratio
    odds_ratio = (a_corr * d_corr) / (b_corr * c_corr)
    
    # Store results
    odds_ratios.append({
        'CUI': cui,
        'Concept': all_cui_names.get(cui, 'Unknown'),
        'Odds Ratio': odds_ratio,
        'Treatment Patients w/ CUI': a,
        'Control Patients w/ CUI': c,
    })

# 4. Create and display a DataFrame sorted by Odds Ratio
df_odds = pd.DataFrame(odds_ratios)
df_odds = df_odds.sort_values(by='Odds Ratio', ascending=False)

print("--- Top 20 Concepts by Odds Ratio (Association with Treatment Group) ---")
# An odds ratio > 1 suggests association with the Treatment group
# An odds ratio < 1 suggests association with the Control group
display(df_odds.head(20))